In [1]:
import pandas as pd

In [2]:
#api setting
generated_time=pd.date_range(start='2016-01-01', end='2025-12-31', freq='D')
time_list = generated_time.strftime('%Y-%m-%d')
url_list=[]
for date in time_list:
    url=f'https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/system-prices/{date}?format=csv'
    url_list.append(url)

In [3]:
dfs=[]
for url in url_list:
    df=pd.read_csv(url)
    if len(df)>0:
        dfs.append(df)
    else:
        print(f'{url} is empty')

In [4]:
from src.extract.save_dir import saved_data_dir

In [5]:
df=pd.concat(dfs)
saved_data_dir('price.csv',df,'../data/raw')
print(df.shape)
print(df.columns)
print(df.head())

price.csv saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\raw\price.csv
(175324, 22)
Index(['SettlementDate', 'SettlementPeriod', 'StartTime', 'CreatedDateTime',
       'SystemSellPrice', 'SystemBuyPrice', 'BsadDefaulted',
       'PriceDerivationCode', 'ReserveScarcityPrice', 'NetImbalanceVolume',
       'SellPriceAdjustment', 'BuyPriceAdjustment', 'ReplacementPrice',
       'ReplacementPriceReferenceVolume', 'TotalAcceptedOfferVolume',
       'TotalAcceptedBidVolume', 'TotalAdjustmentSellVolume',
       'TotalAdjustmentBuyVolume', 'TotalSystemTaggedAcceptedOfferVolume',
       'TotalSystemTaggedAcceptedBidVolume',
       'TotalSystemTaggedAdjustmentSellVolume',
       'TotalSystemTaggedAdjustmentBuyVolume'],
      dtype='str')
  SettlementDate  SettlementPeriod             StartTime  \
0     2016-01-01                 1  2016-01-01T00:00:00Z   
1     2016-01-01                 2  2016-01-01T00:30:00Z   
2     2016-01-01                 3  2016-01-01T01:00:00

In [6]:
from src.transform.half_hour_to_hour import hh_to_hour

In [7]:
#keep the variables I need, adjust the time variable form
keep_cols = [
    "SystemSellPrice",
    "SystemBuyPrice",
    "NetImbalanceVolume",
    "TotalAcceptedOfferVolume",
    "TotalAcceptedBidVolume"
]
hh_to_hour('StartTime',df,keep_cols,'../data/interim/hourly_price.csv')

variable StartTime change to time, saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\interim\hourly_price.csv


In [8]:
df=pd.read_csv('../data/interim/hourly_price.csv')
print(df.shape)
print(df.columns)
print(df.head())
print(df.isna().sum())
print(df.duplicated().sum())

(87668, 6)
Index(['time', 'SystemSellPrice', 'SystemBuyPrice', 'NetImbalanceVolume',
       'TotalAcceptedOfferVolume', 'TotalAcceptedBidVolume'],
      dtype='str')
                        time  SystemSellPrice  SystemBuyPrice  \
0  2016-01-01 00:00:00+00:00        38.868200       38.868200   
1  2016-01-01 01:00:00+00:00        47.511805       47.511805   
2  2016-01-01 02:00:00+00:00        48.500000       48.500000   
3  2016-01-01 03:00:00+00:00        54.469155       54.469155   
4  2016-01-01 04:00:00+00:00        51.368080       51.368080   

   NetImbalanceVolume  TotalAcceptedOfferVolume  TotalAcceptedBidVolume  
0           -98.73185                 693.22315             -1178.27220  
1           128.98310                 795.55835             -1053.93585  
2           125.15745                 665.04005              -926.91595  
3           314.09845                 709.05995              -781.56690  
4           165.72330                 366.46405              -587.41395  

In [9]:
from src.extract.save_dir import saved_data_dir

In [10]:
merge=pd.read_csv('../data/interim/merge_data_demand_output.csv')
supply_pivot=pd.read_csv('../data/interim/hourly_supply_pivot.csv')

supply_pivot['time'] = pd.to_datetime(supply_pivot['time'])
merge['time'] = pd.to_datetime(merge['time'])
final=pd.merge(
    left=supply_pivot,
    right=merge[['time','demand','demand_prediction','error','error_abs','temperature_2m']],
    on='time',
    how='inner',
)

saved_data_dir('part_demand_and_supply.csv',final,'../data/interim/')

part_demand_and_supply.csv saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\interim\part_demand_and_supply.csv


In [11]:
merge_original=pd.read_csv('../data/interim/merged_data.csv')
part_demand_and_supply=pd.read_csv('../data/interim/part_demand_and_supply.csv')

part_demand_and_supply['time'] = pd.to_datetime(part_demand_and_supply['time'])
merge_original['time']=pd.to_datetime(merge_original['time'])

supply_and_demand=pd.merge(
    left=part_demand_and_supply,
    right=merge_original[['time','EMBEDDED_WIND_GENERATION','EMBEDDED_SOLAR_GENERATION','PUMP_STORAGE_PUMPING','IFA_FLOW','BRITNED_FLOW','NEMO_FLOW','NSL_FLOW']],
    on='time',
    how='inner'
)

saved_data_dir('demand_and_supply.csv',supply_and_demand,'../data/interim/')

demand_and_supply.csv saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\interim\demand_and_supply.csv


In [12]:
price=pd.read_csv('../data/interim/hourly_price.csv')
demand_and_supply=pd.read_csv('../data/interim/demand_and_supply.csv')

price['time']=pd.to_datetime(price['time'],utc=True).dt.tz_localize(None)
demand_and_supply['time']=pd.to_datetime(demand_and_supply['time'])

final=pd.merge(
    left=demand_and_supply,
    right=price,
    on='time',
    how='inner'
)

saved_data_dir('final.csv',final,'../data/processed/')

final.csv saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\processed\final.csv
